# 📊 Home Credit — 가설 검증 종합 노트북
## 15개 기본 가설 + 의미 있는 기각 실패 가설

**분석 구조**
- Category 1: 인구통계 (가설 1~4) — 기초 검증
- Category 2: 금융 변수 (가설 5~8) — 반직관 발견
- Category 3: 행동 & 이력 (가설 9~12) — 핵심 시그널
- Category 4: 환경 & 사회 (가설 13~14)
- Category 5: 프로젝트 핵심 가설 (가설 15)
- Category 6: 의미 있는 기각 실패 가설 (Theme 2~5)

**검정 방법**
- 연속형: Mann-Whitney U test + Cohen's d
- 비율 비교: Z-test for proportions
- 3그룹 이상: Kruskal-Wallis test
- 범주형: Chi-square + Cramér's V
- 유의수준: α = 0.05

## 0. 환경 설정 & 데이터 로드

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, norm, chi2_contingency, spearmanr
import os

# 한글 폰트
for fname in ['AppleGothic', 'Apple SD Gothic Neo']:
    try:
        fp = fm.findfont(fm.FontProperties(family=fname), fallback_to_default=False)
        if fp: plt.rcParams['font.family'] = fname; break
    except: pass
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = 'home-credit-default-risk'
OUT_DIR  = '.'
print('라이브러리 로드 완료')

In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────────────
print('데이터 로딩 중...')
df   = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))
prev = pd.read_csv(os.path.join(DATA_DIR, 'previous_application.csv'))
inst = pd.read_csv(os.path.join(DATA_DIR, 'installments_payments.csv'))
bureau = pd.read_csv(os.path.join(DATA_DIR, 'bureau.csv'))

# 파생 변수
df['age']               = -df['DAYS_BIRTH'] / 365
df['employment_years']  = np.where(df['DAYS_EMPLOYED']==365243, np.nan, -df['DAYS_EMPLOYED']/365)
df['ext_source_mean']   = df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)
df['ext_missing']       = df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].isnull().sum(axis=1)
df['credit_income_ratio'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['annuity_ratio']     = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['phone_change_days'] = -df['DAYS_LAST_PHONE_CHANGE']
df['id_publish_days']   = -df['DAYS_ID_PUBLISH']
df['has_realty']        = (df['FLAG_OWN_REALTY']=='Y').astype(int)
df['has_car']           = (df['FLAG_OWN_CAR']=='Y').astype(int)
df['asset_both']        = (df['has_realty'] + df['has_car'] == 2).astype(int)
df['asset_score']       = df['has_realty'] + df['has_car']
edu_map = {'Lower secondary':1,'Secondary / secondary special':2,
           'Incomplete higher':3,'Higher education':4,'Academic degree':5}
df['edu_ord'] = df['NAME_EDUCATION_TYPE'].map(edu_map)

# previous_application 조인
prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_total=('SK_ID_PREV','count'),
    prev_refused=('NAME_CONTRACT_STATUS', lambda x:(x=='Refused').sum())
).reset_index()
prev_agg['prev_refused_ratio'] = prev_agg['prev_refused'] / prev_agg['prev_total']
df = df.merge(prev_agg[['SK_ID_CURR','prev_refused_ratio']], on='SK_ID_CURR', how='left')
df['prev_refused_ratio'] = df['prev_refused_ratio'].fillna(0)

# installments 조인
inst['is_late'] = (inst['DAYS_ENTRY_PAYMENT'] > inst['DAYS_INSTALMENT']).astype(int)
inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_total=('is_late','count'), inst_late=('is_late','sum')).reset_index()
inst_agg['late_payment_ratio'] = inst_agg['inst_late'] / inst_agg['inst_total']
df = df.merge(inst_agg[['SK_ID_CURR','late_payment_ratio']], on='SK_ID_CURR', how='left')
df['late_payment_ratio'] = df['late_payment_ratio'].fillna(0)

# bureau 조인
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    has_bureau=('SK_ID_BUREAU','count'),
    bureau_overdue=('CREDIT_DAY_OVERDUE', lambda x:(x>0).sum())
).reset_index()
bureau_agg['overdue_ratio'] = bureau_agg['bureau_overdue'] / bureau_agg['has_bureau']
df = df.merge(bureau_agg[['SK_ID_CURR','has_bureau','overdue_ratio']], on='SK_ID_CURR', how='left')
df['has_bureau']    = df['has_bureau'].notna().astype(int)
df['overdue_ratio'] = df['overdue_ratio'].fillna(0)
df['had_overdue']   = (df['overdue_ratio'] > 0).astype(int)

# 그룹 라벨
ext_thresh = df['ext_source_mean'].quantile(0.30)
df['risk_A']  = (df['ext_source_mean'] <= ext_thresh).astype(int)
df['risk_B']  = (df['prev_refused_ratio'] > 0.3).astype(int)
df['only_B']  = ((df['risk_A']==0) & (df['risk_B']==1)).astype(int)
df['risk_AB'] = ((df['risk_A']==1) & (df['risk_B']==1)).astype(int)
df['safe']    = ((df['risk_A']==0) & (df['risk_B']==0)).astype(int)

g0 = df[df['TARGET']==0]
g1 = df[df['TARGET']==1]
dr_all = df['TARGET'].mean()

print(f'전체 {len(df):,}명  |  정상 {len(g0):,}  |  연체 {len(g1):,}')
print(f'디폴트율: {dr_all*100:.2f}%')
print(f'only_B: {df["only_B"].sum():,}명  |  A&B: {df["risk_AB"].sum():,}명')

In [ ]:
# ── 공용 함수 ────────────────────────────────────────────────────
def cohens_d(x, y):
    x, y = x.dropna(), y.dropna()
    nx, ny = len(x), len(y)
    pooled = np.sqrt(((nx-1)*x.std()**2 + (ny-1)*y.std()**2) / (nx+ny-2))
    return (x.mean()-y.mean()) / pooled if pooled > 0 else 0

def effect_label(d):
    d = abs(d)
    if d >= 0.8:   return '매우 큰 효과'
    elif d >= 0.5: return '큰 효과'
    elif d >= 0.2: return '중간 효과'
    elif d >= 0.1: return '작은 효과'
    else:          return '실용 무의미 (<0.1)'

def ztest_prop(p1, p2, n1, n2):
    pp = (p1*n1 + p2*n2) / (n1+n2)
    se = np.sqrt(pp*(1-pp)*(1/n1+1/n2))
    z  = (p1-p2)/se if se > 0 else 0
    return z, 2*norm.cdf(-abs(z))

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2, p, _, _ = chi2_contingency(ct)
    n = ct.sum().sum(); r, k = ct.shape
    v = np.sqrt(chi2/n / min(k-1,r-1)) if min(k-1,r-1)>0 else 0
    return v, p, chi2

def verdict(p, d=None):
    if p >= 0.05:
        return f'❌ H0 유지 (기각 실패)  p={p:.4f}'
    elif d is not None and abs(d) < 0.1:
        return f'⚠️  통계 기각 / 실용 무의미  p={p:.4f}, d={abs(d):.3f}'
    else:
        dstr = f', d={abs(d):.3f}' if d is not None else ''
        return f'✅ H0 기각  p={p:.4f}{dstr}'

print('공용 함수 정의 완료')

---
## Category 1 — 인구통계 기초 검증
> **역할**: 분석의 신뢰성 확보 — 상식과 일치하는지 먼저 확인

| 가설 | 검정 방법 |
|------|----------|
| 1. 나이 많을수록 연체율 낮다 | Mann-Whitney + Cohen's d |
| 2. 남성이 여성보다 연체율 높다 | Z-test for proportions |
| 3. 자산 없을수록 연체율 높다 | Kruskal-Wallis |
| 4. 학력 높을수록 연체율 낮다 | Chi-square + Cramér's V |

In [ ]:
# ── 가설 1: 나이 많을수록 연체율 낮다 ──────────────────────────
print('='*60)
print('【가설 1】 나이가 많을수록 연체율이 낮다')
print('  H0: 정상 vs 연체 나이 차이 없음')
print('  H1: 연체 그룹이 더 젊음 (단측)')
print()

a0, a1 = g0['age'].dropna(), g1['age'].dropna()
_, p1 = mannwhitneyu(a0, a1, alternative='greater')
d1 = cohens_d(a0, a1)

df['age_group'] = pd.cut(df['age'], bins=[20,30,40,50,60,75],
                          labels=['20대','30대','40대','50대','60대+'])
age_dr = df.groupby('age_group', observed=True)['TARGET'].mean() * 100

print(f'  정상 평균나이: {a0.mean():.1f}세  |  연체 평균나이: {a1.mean():.1f}세  (차이: {a0.mean()-a1.mean():.1f}세)')
print(f'  Mann-Whitney: p={p1:.2e}  |  Cohen\'s d={abs(d1):.4f}  ({effect_label(d1)})')
print(f'  나이대별 연체율: {dict(age_dr.round(1))}')
print(f'  결과: {verdict(p1, d1)}')

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(a0.clip(20,70), bins=30, color='#3a7bd5', alpha=0.6, label=f'정상(μ={a0.mean():.1f})', density=True)
ax1.hist(a1.clip(20,70), bins=30, color='#e05c5c', alpha=0.6, label=f'연체(μ={a1.mean():.1f})', density=True)
ax1.legend(); ax1.set_xlabel('나이(세)'); ax1.set_title('나이 분포 정상 vs 연체', fontweight='bold')
colors = ['#e05c5c' if v>10 else '#f5a623' if v>8 else '#3a7bd5' for v in age_dr.values]
bars = ax2.bar(age_dr.index, age_dr.values, color=colors, alpha=0.85)
for b, v in zip(bars, age_dr.values): ax2.text(b.get_x()+b.get_width()/2, v+0.1, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax2.axhline(dr_all*100, color='gray', lw=1.5, ls='--', label=f'전체평균 {dr_all*100:.1f}%')
ax2.legend(); ax2.set_ylabel('연체율(%)'); ax2.set_title('나이대별 연체율', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 가설 2: 남성이 여성보다 연체율 높다 ────────────────────────
print('='*60)
print('【가설 2】 남성이 여성보다 연체율이 높다')
print('  H0: 성별 연체율 차이 없음')
print('  H1: 남성 연체율 > 여성 연체율')
print()

mf = df[df['CODE_GENDER'].isin(['M','F'])]
dr_M = mf[mf['CODE_GENDER']=='M']['TARGET'].mean()
dr_F = mf[mf['CODE_GENDER']=='F']['TARGET'].mean()
n_M, n_F = (mf['CODE_GENDER']=='M').sum(), (mf['CODE_GENDER']=='F').sum()
z2, p2 = ztest_prop(dr_M, dr_F, n_M, n_F)

print(f'  남성: {dr_M*100:.2f}% (n={n_M:,})  |  여성: {dr_F*100:.2f}% (n={n_F:,})')
print(f'  차이: +{(dr_M-dr_F)*100:.2f}%p  |  배율: {dr_M/dr_F:.2f}배')
print(f'  Z={z2:.4f}, p={p2:.2e}')
print(f'  결과: {verdict(p2)}')

# 가설 3: 자산
print()
print('='*60)
print('【가설 3】 자산(집/차)이 없을수록 연체율이 높다')
print('  H0: 자산 보유 수와 연체율 관계 없음 (Kruskal-Wallis)')
print('  H1: 자산 없을수록 연체율 높음 (단조 감소)')
print()

asset_dr = df.groupby('asset_score')['TARGET'].mean() * 100
h3, p3 = kruskal(*[df[df['asset_score']==k]['TARGET'].values for k in [0,1,2]])
d3 = cohens_d(df[df['asset_score']==0]['TARGET'].astype(float),
              df[df['asset_score']==2]['TARGET'].astype(float))

print(f'  자산 0개: {asset_dr[0]:.2f}%  |  1개: {asset_dr[1]:.2f}%  |  2개: {asset_dr[2]:.2f}%')
print(f'  Kruskal-Wallis H={h3:.2f}, p={p3:.2e}  |  Cohen\'s d(0 vs 2)={abs(d3):.4f}')
print(f'  결과: {verdict(p3, d3)}')

In [ ]:
# ── 가설 4: 학력 높을수록 안전 ──────────────────────────────────
print('='*60)
print('【가설 4】 학력이 높을수록 연체율이 낮다')
print('  H0: 교육 수준과 연체율 무관')
print('  H1: 학력 높을수록 연체율 낮음')
print()

v4, p4, chi2_4 = cramers_v(df['NAME_EDUCATION_TYPE'].fillna('Unknown'), df['TARGET'])
corr4 = df[['edu_ord','TARGET']].dropna().corr().iloc[0,1]
edu_dr = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean()*100

print(f'  Chi-square={chi2_4:.1f}  |  Cramér\'s V={v4:.4f}  |  p={p4:.2e}')
print(f'  학력-연체율 상관: r={corr4:.4f}')
print('  학력별 연체율:')
for edu, v in edu_dr.sort_values().items():
    print(f'    {edu}: {v:.1f}%')
print(f'  결과: {verdict(p4)}')

# 시각화 (가설 2,3,4)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# 성별
axes[0].bar(['여성(F)','남성(M)'], [dr_F*100, dr_M*100], color=['#3a7bd5','#e05c5c'], alpha=0.85)
for bar, v in zip(axes[0].patches, [dr_F*100, dr_M*100]):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.1, f'{v:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('가설2: 성별 연체율', fontweight='bold'); axes[0].set_ylabel('연체율(%)')
# 자산
axes[1].bar(['없음(0)','1개','2개'], asset_dr.values, color=['#e05c5c','#f5a623','#3a7bd5'], alpha=0.85)
for bar, v in zip(axes[1].patches, asset_dr.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.05, f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('가설3: 자산 보유수별 연체율', fontweight='bold')
# 학력
edu_sorted = edu_dr.sort_values(ascending=False)
axes[2].barh([e[:15] for e in edu_sorted.index], edu_sorted.values, color='#8855cc', alpha=0.85)
axes[2].set_title('가설4: 학력별 연체율', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Category 2 — 금융 변수 (반직관 발견)
> **역할**: 기존 통념이 틀렸음을 증명 — 소득·DTI 약함, EXT 최강

| 가설 | 검정 | 반직관 포인트 |
|------|------|---------------|
| 5. 소득 높을수록 안전 | Mann-Whitney | d=0.015로 효과 거의 없음 |
| 6. DTI 높을수록 위험 | 분위 분석 | 역U자형 — 중간이 제일 위험 |
| 7. 상환부담 클수록 위험 | Mann-Whitney | d=0.052 약한 효과 |
| 8. EXT_SOURCE가 최강 시그널 | Cohen's d 비교 | d=0.836, 2위(나이)의 2.9배 |

In [ ]:
# ── 가설 5~8 ────────────────────────────────────────────────────

# 가설 5: 소득
i0, i1 = g0['AMT_INCOME_TOTAL'].dropna(), g1['AMT_INCOME_TOTAL'].dropna()
_, p5 = mannwhitneyu(i0, i1, alternative='two-sided')
d5 = cohens_d(i0, i1)
df['income_decile'] = pd.qcut(df['AMT_INCOME_TOTAL'], 10, labels=False) + 1
inc_dr = df.groupby('income_decile')['TARGET'].mean() * 100

print('【가설 5】 소득 높을수록 연체율 낮다')
print(f'  정상 중앙값: {i0.median():,.0f}  |  연체 중앙값: {i1.median():,.0f}')
print(f'  소득 1분위: {inc_dr.iloc[0]:.1f}%  →  10분위: {inc_dr.iloc[-1]:.1f}%  (격차: {inc_dr.iloc[0]-inc_dr.iloc[-1]:.1f}%p)')
print(f'  p={p5:.2e}  |  d={abs(d5):.4f}')
print(f'  결과: {verdict(p5, d5)}')
print(f'  → 통계적으로 유의하나 d={abs(d5):.3f}로 실질적 의미 없음!')

# 가설 6: DTI
dti_clip = df['credit_income_ratio'].clip(upper=20)
df['dti_decile'] = pd.qcut(dti_clip, 10, labels=False, duplicates='drop') + 1
dti_dr = df.groupby('dti_decile')['TARGET'].mean() * 100
monotone = all(dti_dr.iloc[i]<=dti_dr.iloc[i+1] for i in range(len(dti_dr)-1))

print()
print('【가설 6】 DTI 높을수록 위험하다 (역U자형 검증)')
print(f'  DTI 1분위:{dti_dr.iloc[0]:.1f}%  |  피크:{dti_dr.idxmax()}분위:{dti_dr.max():.1f}%  |  10분위:{dti_dr.iloc[-1]:.1f}%')
print(f'  단조증가 패턴: {"✅" if monotone else "❌ 역U자형 확인!"}')
print(f'  결과: {"❌ 단조증가 가설 기각 — 역U자형 입증" if not monotone else "✅ 단조증가"}')

# 가설 7: 상환부담
an0, an1 = g0['annuity_ratio'].dropna(), g1['annuity_ratio'].dropna()
_, p7 = mannwhitneyu(an0, an1, alternative='less')
d7 = cohens_d(an0, an1)

print()
print('【가설 7】 연간 상환 부담 클수록 연체율 높다')
print(f'  정상 중앙값: {an0.median():.4f}  |  연체: {an1.median():.4f}  |  d={abs(d7):.4f}')
print(f'  결과: {verdict(p7, d7)}')

# 가설 8: EXT_SOURCE 시그널 강도 비교
vars8 = {'EXT_SOURCE 평균':'ext_source_mean','나이':'age','과거거절비율':'prev_refused_ratio',
         '재직기간':'employment_years','할부연체비율':'late_payment_ratio',
         '휴대폰변경일':'phone_change_days','DTI':'credit_income_ratio','연간소득':'AMT_INCOME_TOTAL'}
d8 = {lbl: abs(cohens_d(g0[var].dropna(), g1[var].dropna())) for lbl,var in vars8.items()}
d8_sorted = sorted(d8.items(), key=lambda x: x[1], reverse=True)

print()
print('【가설 8】 EXT_SOURCE가 단일 변수 중 가장 강한 시그널이다')
print(f'  순위  변수                  Cohen\'s d')
print('  ' + '-'*40)
for rank, (lbl, dv) in enumerate(d8_sorted, 1):
    mark = ' ← 1위 ⭐' if rank==1 else ''
    print(f'  {rank}위   {lbl:<20}  {dv:.4f}{mark}')
is_top = d8_sorted[0][0] == 'EXT_SOURCE 평균'
print(f'  결과: {"✅ EXT_SOURCE 1위 확인" if is_top else "❌ 기각 실패"}')

In [ ]:
# ── 가설 5~8 시각화 ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 소득 분위별
axes[0].plot(inc_dr.index, inc_dr.values, 'o-', color='#8855cc', lw=2)
axes[0].fill_between(inc_dr.index, inc_dr.values, alpha=0.15, color='#8855cc')
axes[0].axhline(dr_all*100, color='gray', lw=1.5, ls='--')
axes[0].set_title(f'가설5: 소득 분위별 연체율\n(격차 고작 {inc_dr.iloc[0]-inc_dr.iloc[-1]:.1f}%p!)', fontweight='bold')
axes[0].set_xlabel('소득 분위 (1=최하, 10=최상)')

# DTI 역U자형
peak = dti_dr.idxmax()
colors_dti = ['#e05c5c' if i==peak else '#3a7bd5' for i in dti_dr.index]
axes[1].bar(dti_dr.index, dti_dr.values, color=colors_dti, alpha=0.85)
axes[1].set_title('가설6: DTI 분위별 연체율\n(역U자형 — 중간이 제일 위험!)', fontweight='bold')
axes[1].set_xlabel('DTI 분위')

# Cohen's d 비교
lbls = [x[0] for x in d8_sorted]
vals = [x[1] for x in d8_sorted]
clrs = ['#e05c5c' if l=='EXT_SOURCE 평균' else '#3a7bd5' if l in ['나이','재직기간'] else '#888' for l in lbls]
axes[2].barh(lbls[::-1], vals[::-1], color=clrs[::-1], alpha=0.85)
axes[2].axvline(0.2, color='orange', lw=1, ls='--', label='중간효과(0.2)')
axes[2].legend(fontsize=8)
axes[2].set_title('가설8: 변수별 Cohen\'s d 순위', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Category 3 — 행동 & 이력 변수
> **역할**: 실제로 강한 예측 시그널 확정

| 가설 | 핵심 발견 |
|------|----------|
| 9. EXT 결측 = 위험 신호 | 결측 0→2개: 7.3%→9.9% 단조 증가 |
| 10. 과거 할부 연체 → 현재 연체 | d=0.260, 단조 증가 |
| 11. 타기관 이력 없으면 위험 | 10.1% vs 7.7% |
| 12. 과거 연체 이력 → 현재도 위험 | 15.9% vs 8.0% (2배) |

In [ ]:
# ── 가설 9~12 ───────────────────────────────────────────────────

# 가설 9: EXT 결측 = 위험 신호
miss_dr = df.groupby('ext_missing')['TARGET'].agg(['mean','count'])
miss_dr.columns = ['연체율','인원']
groups9 = [df[df['ext_missing']==k]['TARGET'].values for k in range(3)]
h9, p9  = kruskal(*groups9)

print('【가설 9】 EXT_SOURCE 결측 많을수록 연체율 높다 (결측 = 위험 신호)')
for i, row in miss_dr.iterrows():
    print(f'  결측 {i}개: {row["연체율"]*100:.2f}% ({row["인원"]:,.0f}명)')
print(f'  Kruskal-Wallis H={h9:.2f}, p={p9:.2e}')
print(f'  결과: {verdict(p9)}')
print(f'  → 점수 없음 = 신용기관에 기록 없음 = 검증 안됨 = 위험!')

# 가설 10: 할부 연체
lt0, lt1 = g0['late_payment_ratio'].dropna(), g1['late_payment_ratio'].dropna()
_, p10 = mannwhitneyu(lt0, lt1, alternative='less')
d10 = cohens_d(lt0, lt1)
df['late_q'] = pd.qcut(df['late_payment_ratio'], 5, duplicates='drop')
late_dr = df.groupby('late_q', observed=True)['TARGET'].mean() * 100

print()
print('【가설 10】 과거 할부 연체 경험 → 현재 연체율 높다')
print(f'  정상 평균: {lt0.mean():.4f}  |  연체 평균: {lt1.mean():.4f}')
print(f'  분위별: ', end='')
for v in late_dr.values: print(f'{v:.1f}%', end=' → ')
print(f'\n  p={p10:.2e}  |  d={abs(d10):.4f} ({effect_label(d10)})')
print(f'  결과: {verdict(p10, d10)}')

# 가설 11: bureau 이력
dr_no  = df[df['has_bureau']==0]['TARGET'].mean()
dr_yes = df[df['has_bureau']==1]['TARGET'].mean()
n_no, n_yes = (df['has_bureau']==0).sum(), (df['has_bureau']==1).sum()
z11, p11 = ztest_prop(dr_no, dr_yes, n_no, n_yes)

print()
print('【가설 11】 타기관 신용 이력 없으면 연체율 더 높다')
print(f'  이력 없음: {dr_no*100:.2f}% ({n_no:,}명)  |  이력 있음: {dr_yes*100:.2f}% ({n_yes:,}명)')
print(f'  Z={z11:.4f}, p={p11:.2e}')
print(f'  결과: {verdict(p11)}')

# 가설 12: 과거 연체
dr_ov  = df[df['had_overdue']==1]['TARGET'].mean()
dr_nov = df[df['had_overdue']==0]['TARGET'].mean()
n_ov, n_nov = (df['had_overdue']==1).sum(), (df['had_overdue']==0).sum()
z12, p12 = ztest_prop(dr_ov, dr_nov, n_ov, n_nov)

print()
print('【가설 12】 타기관 과거 연체 이력 → 현재도 연체 위험')
print(f'  과거 연체 있음: {dr_ov*100:.2f}% ({n_ov:,}명)  |  없음: {dr_nov*100:.2f}% ({n_nov:,}명)')
print(f'  Z={z12:.4f}, p={p12:.2e}')
print(f'  결과: {verdict(p12)}')

---
## Category 4~5 — 환경·사회 & 프로젝트 핵심 가설
> **가설 15가 이 프로젝트의 핵심 주장을 직접 증명**

In [ ]:
# ── 가설 13~14: 환경 ────────────────────────────────────────────

# 가설 13: 주변 연체자
sc0, sc1 = g0['DEF_60_CNT_SOCIAL_CIRCLE'].dropna(), g1['DEF_60_CNT_SOCIAL_CIRCLE'].dropna()
_, p13 = mannwhitneyu(sc0, sc1, alternative='less')
d13 = cohens_d(sc0, sc1)
df['social_g'] = df['DEF_60_CNT_SOCIAL_CIRCLE'].clip(0,3).fillna(0).astype(int)
social_dr = df.groupby('social_g')['TARGET'].mean() * 100

print('【가설 13】 주변 연체자 많을수록 본인도 연체율 높다')
print(f'  주변 연체자 수별 연체율: {dict(social_dr.round(1))}')
print(f'  d={abs(d13):.4f} ({effect_label(d13)})  |  p={p13:.2e}')
print(f'  결과: {verdict(p13, d13)}')

# 가설 14: 지역
reg_dr = df.groupby('REGION_RATING_CLIENT_W_CITY')['TARGET'].mean() * 100
h14, p14 = kruskal(*[df[df['REGION_RATING_CLIENT_W_CITY']==k]['TARGET'].values for k in [1,2,3]])

print()
print('【가설 14】 지역 신용등급 낮은 지역 = 연체율 높다')
for k, v in reg_dr.items():
    n = (df['REGION_RATING_CLIENT_W_CITY']==k).sum()
    print(f'  등급 {k}: {v:.1f}%  ({n:,}명)')
print(f'  Kruskal-Wallis H={h14:.2f}, p={p14:.2e}')
print(f'  결과: {verdict(p14)}')

print()
print('='*60)

# 가설 15: 핵심 가설
g_onlyB = df[df['only_B']==1]
g_AB    = df[df['risk_AB']==1]

p1_ob = g_onlyB['TARGET'].mean()
p2_ab = g_AB['TARGET'].mean()
z15, p15 = ztest_prop(p1_ob, p2_ab, len(g_onlyB), len(g_AB))

# EXT 통제 후 비교
ab_q75 = g_AB['ext_source_mean'].quantile(0.75)
ob_q25 = g_onlyB['ext_source_mean'].quantile(0.25)
overlap_AB   = g_AB[(g_AB['ext_source_mean']>=ob_q25-0.05)&(g_AB['ext_source_mean']<=ab_q75+0.05)]
overlap_onlyB= g_onlyB[(g_onlyB['ext_source_mean']>=ob_q25-0.05)&(g_onlyB['ext_source_mean']<=ab_q75+0.05)]
z15b, p15b = ztest_prop(overlap_onlyB['TARGET'].mean(), overlap_AB['TARGET'].mean(),
                         len(overlap_onlyB), len(overlap_AB))

print('【가설 15】 only_B와 A∩B의 연체율 차이는 EXT만으로 설명 안된다')
print('  H0: EXT 통제 시 only_B = A∩B 연체율')
print(f'  전체 비교:  only_B={p1_ob*100:.2f}%  vs  A∩B={p2_ab*100:.2f}%  (Z={z15:.2f}, p={p15:.2e})')
print(f'  EXT 통제 후: only_B={overlap_onlyB["TARGET"].mean()*100:.2f}%  vs  A∩B={overlap_AB["TARGET"].mean()*100:.2f}%  (Z={z15b:.2f}, p={p15b:.2e})')
print(f'  결과: {verdict(p15b)} — EXT 통제 후에도 차이 존재')
print()
print(f'  ★★ 결론: only_B(회복형)과 A∩B(고위험)은')
print(f'           EXT 점수 차이뿐 아니라 구조적으로 다른 그룹이다')

---
## Category 6 — 의미 있는 기각 실패 가설
> **핵심**: '차이 없다'는 것도 중요한 발견 — 정책 변경의 근거

| 가설 | 결과 | 비즈니스 의미 |
|------|------|---------------|
| A. EXT 좋으면 거절 횟수 무관 | ⚠️ d=0.084 | 현재 신용이 과거를 덮어씀 |
| B. 장기재직자는 소득 차이 무관 | ⚠️ d=0.044 | 직업 안정성 = 소득 대체 |
| C. 우량 조건에서 DTI 무의미 | ⚠️ d=0.022 | 우량 고객에게 DTI 기준 완화 |
| D. 자산 우량 그룹에서 성별 차이 축소 | ⚠️ d=0.080 | 충분한 조건에서 성별 불필요 |
| E. only_B 내에서 나이 차이 없음 | ⚠️ d=0.059 | 회복형 신상품 연령 제한 불필요 |

In [ ]:
# ── 기각 실패 가설 A~E ──────────────────────────────────────────

results_null = []

# A. EXT 좋으면 거절 비율 무의미
ext_good = df[df['ext_source_mean'] > df['ext_source_mean'].quantile(0.30)]
rA_hi = ext_good[ext_good['prev_refused_ratio']> 0.5]['TARGET']
rA_lo = ext_good[(ext_good['prev_refused_ratio']>0.3)&(ext_good['prev_refused_ratio']<=0.5)]['TARGET']
zA, pA = ztest_prop(rA_hi.mean(), rA_lo.mean(), len(rA_hi), len(rA_lo))
dA = cohens_d(rA_hi.astype(float), rA_lo.astype(float))
results_null.append(('A. EXT양호+거절 50%↑ vs 30~50%', rA_hi.mean()*100, rA_lo.mean()*100, pA, dA,
                     '현재 신용이 좋으면 거절 횟수 무관'))

# B. 장기재직자 소득 차이
le = df[df['employment_years'] >= 10]
med_inc = le['AMT_INCOME_TOTAL'].median()
leB_lo = le[le['AMT_INCOME_TOTAL'] <  med_inc]['TARGET']
leB_hi = le[le['AMT_INCOME_TOTAL'] >= med_inc]['TARGET']
zB, pB = ztest_prop(leB_lo.mean(), leB_hi.mean(), len(leB_lo), len(leB_hi))
dB = cohens_d(leB_lo.astype(float), leB_hi.astype(float))
results_null.append(('B. 재직10년+ 저소득 vs 고소득', leB_lo.mean()*100, leB_hi.mean()*100, pB, dB,
                     '직업 안정성이 소득 격차를 상쇄'))

# C. 우량 조건에서 DTI 무의미
stable = df[(df['ext_source_mean']>df['ext_source_mean'].quantile(0.30))&(df['employment_years']>=5)]
med_dti = stable['credit_income_ratio'].quantile(0.5)
stC_lo = stable[stable['credit_income_ratio'] <  med_dti]['TARGET']
stC_hi = stable[stable['credit_income_ratio'] >= med_dti]['TARGET']
zC, pC = ztest_prop(stC_lo.mean(), stC_hi.mean(), len(stC_lo), len(stC_hi))
dC = cohens_d(stC_lo.astype(float), stC_hi.astype(float))
results_null.append(('C. EXT양호+장기재직 저DTI vs 고DTI', stC_lo.mean()*100, stC_hi.mean()*100, pC, dC,
                     '우량 고객에게 DTI 기준 강화 불필요'))

# D. 자산 우량 그룹 성별
prem = df[(df['asset_both']==1)&(df['ext_source_mean']>df['ext_source_mean'].quantile(0.30))&
          (df['CODE_GENDER'].isin(['M','F']))]
pmD = prem[prem['CODE_GENDER']=='M']['TARGET']
pfD = prem[prem['CODE_GENDER']=='F']['TARGET']
zD, pD = ztest_prop(pmD.mean(), pfD.mean(), len(pmD), len(pfD))
dD = cohens_d(pmD.astype(float), pfD.astype(float))
results_null.append(('D. 자산우량 남성 vs 여성', pmD.mean()*100, pfD.mean()*100, pD, dD,
                     f'전체 성별 차이 3.14%p → {abs(pmD.mean()-pfD.mean())*100:.2f}%p로 축소'))

# E. only_B 내 나이
ob = df[df['only_B']==1]
obE_y = ob[ob['age'] < 40]['TARGET']
obE_o = ob[ob['age'] >= 40]['TARGET']
zE, pE = ztest_prop(obE_y.mean(), obE_o.mean(), len(obE_y), len(obE_o))
dE = cohens_d(obE_y.astype(float), obE_o.astype(float))
results_null.append(('E. 회복형(only_B) 40세미만 vs 40세+', obE_y.mean()*100, obE_o.mean()*100, pE, dE,
                     '회복형 신상품 연령 제한 불필요'))

# 결과 출력
print('='*72)
print('의미 있는 기각 실패 / 실용 무의미 가설 결과')
print('='*72)
print(f'{"가설":<35} {"그룹1":>8} {"그룹2":>8} {"Cohen_d":>9} {"판정":<10} 비즈니스 의미')
print('-'*90)
for lbl, v1, v2, p, d, msg in results_null:
    verd = '❌ 기각실패' if p>=0.05 else ('⚠️ 실용무의미' if abs(d)<0.1 else '✅ 기각')
    print(f'{lbl:<35} {v1:>7.2f}% {v2:>7.2f}% {abs(d):>9.4f} {verd:<12} {msg}')

In [ ]:
# ── 기각 실패 시각화 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 좌: Cohen's d 비교 (실용 무의미 강조)
lbls_null = [r[0][:20] for r in results_null]
ds_null   = [abs(r[4]) for r in results_null]
colors_n  = ['#cccccc' if d<0.1 else '#3a7bd5' for d in ds_null]
bars_n = axes[0].barh(lbls_null[::-1], ds_null[::-1], color=colors_n[::-1], alpha=0.85)
axes[0].axvline(0.1, color='#e05c5c', lw=2, ls='--', label='실용 무의미 기준 (d=0.1)')
axes[0].axvline(0.2, color='#f5a623', lw=1.5, ls=':', label='작은 효과 기준 (d=0.2)')
axes[0].legend(fontsize=9)
axes[0].set_title('기각 실패 가설들의 Cohen\'s d\n(기준선 왼쪽 = 실용적 의미 없음)', fontweight='bold')

# 우: 의미 시각화 — 각 조건에서 연체율 차이 vs 전체 평균
case_labels = ['전체\n성별차이', '자산우량\n성별차이', '전체\n소득차이', '장기재직\n소득차이',
               '전체\nDTI차이', '우량조건\nDTI차이']
case_diffs  = [
    abs(df[df['CODE_GENDER']=='M']['TARGET'].mean() - df[df['CODE_GENDER']=='F']['TARGET'].mean()) * 100,
    abs(pmD.mean() - pfD.mean()) * 100,
    abs(df[df['income_decile']==1]['TARGET'].mean() - df[df['income_decile']==10]['TARGET'].mean()) * 100,
    abs(leB_lo.mean() - leB_hi.mean()) * 100,
    abs(df[df['dti_decile']==6]['TARGET'].mean() - df[df['dti_decile']==1]['TARGET'].mean()) * 100,
    abs(stC_lo.mean() - stC_hi.mean()) * 100,
]
clrs2 = ['#e05c5c','#f5a623','#e05c5c','#f5a623','#e05c5c','#f5a623']
axes[1].bar(case_labels, case_diffs, color=clrs2, alpha=0.85)
for bar, v in zip(axes[1].patches, case_diffs):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}%p', ha='center', fontsize=8.5, fontweight='bold')
axes[1].set_ylabel('연체율 차이 (%p)')
axes[1].set_title('조건 통제 전후 연체율 차이 비교\n(주황=조건통제후, 빨강=전체)', fontweight='bold')

plt.tight_layout()
plt.show()
print('시각화 완료')

---
## 종합 결론 — 가설 검증이 말하는 것

### 15개 가설의 역할

```
[가설 1~4] 기초 검증 통과 → 분석 신뢰성 확보
     ↓
[가설 5~8] 기존 통념 반박 → 소득·DTI 약함, EXT 최강
     ↓
[가설 9~12] 핵심 시그널 확정 → 이력·행동 변수 유효성
     ↓
[가설 13~14] 환경 변수 확인 → 지역·사회적 맥락
     ↓
[가설 15] 프로젝트 핵심 주장 직접 증명
     ↓
[기각 실패] 정책 변경 근거 제공
     ↓
최종 메시지: 단일 기준 심사는 28,211명의 우량 고객을 잘못 거절한다
```

### 기각 실패 가설이 말하는 정책 제안

| 발견 | 현재 정책 | 제안 정책 |
|------|-----------|----------|
| EXT 좋으면 거절 횟수 무관 | 거절 이력 → 위험 처리 | EXT 기준 통과 시 거절 이력 가중치 제거 |
| 장기재직 = 소득 대체 | 소득 기준 심사 | 재직 10년+ 는 소득 기준 완화 |
| 우량 조건에서 DTI 무의미 | DTI 임계값 엄격 적용 | 우량 고객 DTI 기준 완화 |
| 자산 우량 시 성별 차이 축소 | 성별 가중치 적용 | 자산+신용 우량 시 성별 중립 심사 |

In [ ]:
# ── 최종 요약 출력 ───────────────────────────────────────────────
print('='*68)
print('📊 가설 검증 종합 요약')
print('='*68)

summary = [
    ('1', '나이 많을수록 안전',           '✅ 기각', 'd=0.288 중간 효과'),
    ('2', '남성 > 여성 연체율',            '✅ 기각', 'Z=30.3, 1.45배'),
    ('3', '자산 없을수록 위험',            '✅ 기각', 'H=123.9'),
    ('4', '학력 높을수록 안전',            '✅ 기각', 'V=0.058'),
    ('5', '소득 높을수록 안전',            '⚠️ 실용무의미', 'd=0.015 격차 2.1%p'),
    ('6', 'DTI 높을수록 위험',             '❌ 역U자형 확인', '중간분위가 피크'),
    ('7', '상환부담 클수록 위험',          '⚠️ 실용무의미', 'd=0.052'),
    ('8', 'EXT_SOURCE 단일 최강 시그널',   '✅ 기각', 'd=0.836, 1위'),
    ('9', 'EXT 결측 = 위험신호',          '✅ 기각', '7.3%→9.9% 단조증가'),
    ('10','과거 할부연체→현재 위험',       '✅ 기각', 'd=0.260'),
    ('11','신용이력 없으면 더 위험',       '✅ 기각', '10.1% vs 7.7%'),
    ('12','과거 연체이력→현재 위험',       '✅ 기각', '15.9% vs 8.0%'),
    ('13','주변 연체자 많을수록 위험',     '✅ 기각', 'd=0.115'),
    ('14','나쁜 지역 = 연체율 높음',      '✅ 기각', '4.8%→11.4%'),
    ('15','only_B vs A∩B 구조적 차이',    '✅ 기각', 'EXT 통제 후도 차이'),
    ('A', 'EXT 좋으면 거절횟수 무관',     '⚠️ 실용무의미', 'd=0.084'),
    ('B', '장기재직자 소득 차이 무관',    '⚠️ 실용무의미', 'd=0.044'),
    ('C', '우량조건에서 DTI 무의미',      '⚠️ 실용무의미', 'd=0.022'),
    ('D', '자산우량 그룹 성별 차이 축소', '⚠️ 실용무의미', 'd=0.080'),
    ('E', '회복형 내 나이 차이 없음',     '⚠️ 실용무의미', 'd=0.059'),
]

print(f'  {"#":<4} {"가설 요약":<28} {"결과":<18} {"핵심 수치"}')
print('  ' + '-'*70)
for num, desc, res, key in summary:
    print(f'  {num:<4} {desc:<28} {res:<18} {key}')

print()
print('★ 핵심 결론:')
print(f'  전체 디폴트율: {dr_all*100:.2f}%')
print(f'  only_B 연체율: {df["only_B"].pipe(lambda s: df[s==1]["TARGET"].mean())*100:.2f}%  (평균 이하!)')
print(f'  A∩B 연체율:    {df["risk_AB"].pipe(lambda s: df[s==1]["TARGET"].mean())*100:.2f}%  (평균 2.6배!)')
print(f'  격차:          {df[df["risk_AB"]==1]["TARGET"].mean()/df[df["only_B"]==1]["TARGET"].mean():.1f}배')